## 1. Montar Drive e criar pastas

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

PROJECT = '/content/drive/MyDrive/gtzan_projeto'

pastas = [
    'data/raw',
    'data/processed/tabular',
    'data/processed/images',
    'modelos',
    'resultados',
]
for p in pastas:
    os.makedirs(f'{PROJECT}/{p}', exist_ok=True)

print('Drive montado. Projeto em:', PROJECT)
print('Pastas criadas:')
for p in pastas:
    print(f'  {PROJECT}/{p}')

Mounted at /content/drive
Drive montado. Projeto em: /content/drive/MyDrive/gtzan_projeto
Pastas criadas:
  /content/drive/MyDrive/gtzan_projeto/data/raw
  /content/drive/MyDrive/gtzan_projeto/data/processed/tabular
  /content/drive/MyDrive/gtzan_projeto/data/processed/images
  /content/drive/MyDrive/gtzan_projeto/modelos
  /content/drive/MyDrive/gtzan_projeto/resultados


## 2. Download do dataset

In [4]:
import os

# configura credenciais do Kaggle
os.makedirs('/root/.kaggle', exist_ok=True)

ret = os.system('cp /content/kaggle.json /root/.kaggle/')
assert ret == 0, 'ERRO: kaggle.json não encontrado em /content/. Faça o upload primeiro.'

os.system('chmod 600 /root/.kaggle/kaggle.json')

RAW = f'{PROJECT}/data/raw'

# usando ! para ver erros reais do kaggle CLI
print('Iniciando download — pode demorar alguns minutos...')
!kaggle datasets download andradaolteanu/gtzan-dataset-music-genre-classification -p {RAW} --unzip

# confirma que os arquivos chegaram
print('\nConteúdo de data/raw:')
for item in sorted(os.listdir(RAW)):
    print(f'  {item}')

csv_path = f'{RAW}/Data/features_30_sec.csv'
img_dir  = f'{RAW}/Data/images_original'
assert os.path.exists(csv_path), f'ERRO: {csv_path} não encontrado'
assert os.path.isdir(img_dir),   f'ERRO: {img_dir} não encontrado'

print('\nDownload OK. CSV e imagens encontrados.')

Iniciando download — pode demorar alguns minutos...
Dataset URL: https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification
License(s): other
100% 1.21G/1.21G [00:10<00:00, 120MB/s]


Conteúdo de data/raw:
  Data

Download OK. CSV e imagens encontrados.


## 3. Processar dados tabulares

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

csv_path = f'{RAW}/Data/features_30_sec.csv'
assert os.path.exists(csv_path), f'ERRO: arquivo não encontrado em {csv_path}. Rode a Célula 2 primeiro.'

# carrega e inspeciona
df = pd.read_csv(csv_path)
print(f'Shape original: {df.shape}')
print(f'Primeiras colunas: {list(df.columns[:6])} ...')
print(f'\nDistribuição de gêneros:')
print(df['label'].value_counts().sort_index().to_string())

# prepara X e y
df = df.drop(columns=['filename', 'length'])
X = df.drop(columns=['label'])
y = df['label']
print(f'\nFeatures: {X.shape[1]} colunas')
print(f'Amostras: {len(X)}')

# split estratificado 70 / 15 / 15
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'\nTamanhos após split:')
print(f'  Treino : {len(X_train)} amostras')
print(f'  Val    : {len(X_val)} amostras')
print(f'  Teste  : {len(X_test)} amostras')
print(f'\nDistribuição no treino (deve ser uniforme ~70 por gênero):')
print(y_train.value_counts().sort_index().to_string())

# normalização DEPOIS do split
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print(f'\nVerificação do scaler (treino):')
print(f'  Média  : {X_train_sc.mean():.6f}  (deve ser ~0)')
print(f'  Std    : {X_train_sc.std():.6f}   (deve ser ~1)')

# salva
TAB = f'{PROJECT}/data/processed/tabular'
np.save(f'{TAB}/X_train.npy', X_train_sc)
np.save(f'{TAB}/X_val.npy',   X_val_sc)
np.save(f'{TAB}/X_test.npy',  X_test_sc)
np.save(f'{TAB}/y_train.npy', y_train.values)
np.save(f'{TAB}/y_val.npy',   y_val.values)
np.save(f'{TAB}/y_test.npy',  y_test.values)

print('\nArquivos salvos em data/processed/tabular/:')
for fname in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test']:
    arr = np.load(f'{TAB}/{fname}.npy', allow_pickle=True)
    print(f'  {fname}.npy  → shape {arr.shape}  dtype {arr.dtype}')

print('\nDados tabulares prontos.')

Shape original: (1000, 60)
Primeiras colunas: ['filename', 'length', 'chroma_stft_mean', 'chroma_stft_var', 'rms_mean', 'rms_var'] ...

Distribuição de gêneros:
label
blues        100
classical    100
country      100
disco        100
hiphop       100
jazz         100
metal        100
pop          100
reggae       100
rock         100

Features: 57 colunas
Amostras: 1000

Tamanhos após split:
  Treino : 700 amostras
  Val    : 150 amostras
  Teste  : 150 amostras

Distribuição no treino (deve ser uniforme ~70 por gênero):
label
blues        70
classical    70
country      70
disco        70
hiphop       70
jazz         70
metal        70
pop          70
reggae       70
rock         70

Verificação do scaler (treino):
  Média  : -0.000000  (deve ser ~0)
  Std    : 1.000000   (deve ser ~1)

Arquivos salvos em data/processed/tabular/:
  X_train.npy  → shape (700, 57)  dtype float64
  X_val.npy  → shape (150, 57)  dtype float64
  X_test.npy  → shape (150, 57)  dtype float64
  y_train.npy  

## 4. Processar espectrogramas

In [8]:
from PIL import Image
import pathlib
import re
import numpy as np
from sklearn.model_selection import train_test_split

IMG_DIR = pathlib.Path(f'{RAW}/Data/images_original')
assert IMG_DIR.is_dir(), f'ERRO: diretório não encontrado: {IMG_DIR}'

all_images = sorted(IMG_DIR.glob('**/*.png'))
print(f'Imagens encontradas: {len(all_images)}')

# mostra exemplos do nome dos arquivos para confirmar o padrão
print('Exemplos de nomes:', [p.name for p in all_images[:3]])

def load_img(path):
    return np.array(Image.open(path).convert('L').resize((128, 128))) / 255.0

def get_label(path):
    m = re.match(r'([a-z_]+)\d+\.png', path.name)
    assert m is not None, f'Nome inesperado: {path.name}'
    return m.group(1)

# carrega tudo
print('Carregando imagens...')
images = np.array([load_img(p) for p in all_images])
labels = np.array([get_label(p) for p in all_images])

print(f'Shape do array de imagens: {images.shape}')  # (1000, 128, 128)
print(f'Gêneros únicos: {sorted(set(labels))}')
unique, counts = np.unique(labels, return_counts=True)
print('Distribuição:')
for g, c in zip(unique, counts):
    print(f'  {g}: {c}')

# split com os mesmos parâmetros de 3
idx = np.arange(len(images))
idx_tr, idx_tmp = train_test_split(idx, test_size=0.30, random_state=42, stratify=labels)
idx_val, idx_te = train_test_split(idx_tmp, test_size=0.50, random_state=42, stratify=labels[idx_tmp])

print(f'\nTamanhos após split:')
print(f'  Treino : {len(idx_tr)}')
print(f'  Val    : {len(idx_val)}')
print(f'  Teste  : {len(idx_te)}')

# salva
IMG = f'{PROJECT}/data/processed/images'
np.save(f'{IMG}/X_train.npy', images[idx_tr])
np.save(f'{IMG}/X_val.npy',   images[idx_val])
np.save(f'{IMG}/X_test.npy',  images[idx_te])
np.save(f'{IMG}/y_train.npy', labels[idx_tr])
np.save(f'{IMG}/y_val.npy',   labels[idx_val])
np.save(f'{IMG}/y_test.npy',  labels[idx_te])

print('\nArquivos salvos em data/processed/images/:')
for fname in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test']:
    arr = np.load(f'{IMG}/{fname}.npy', allow_pickle=True)
    print(f'  {fname}.npy  → shape {arr.shape}  dtype {arr.dtype}')

print('\n Espectrogramas prontos.')

Imagens encontradas: 999
Exemplos de nomes: ['blues00000.png', 'blues00001.png', 'blues00002.png']
Carregando imagens...
Shape do array de imagens: (999, 128, 128)
Gêneros únicos: [np.str_('blues'), np.str_('classical'), np.str_('country'), np.str_('disco'), np.str_('hiphop'), np.str_('jazz'), np.str_('metal'), np.str_('pop'), np.str_('reggae'), np.str_('rock')]
Distribuição:
  blues: 100
  classical: 100
  country: 100
  disco: 100
  hiphop: 100
  jazz: 99
  metal: 100
  pop: 100
  reggae: 100
  rock: 100

Tamanhos após split:
  Treino : 699
  Val    : 150
  Teste  : 150

Arquivos salvos em data/processed/images/:
  X_train.npy  → shape (699, 128, 128)  dtype float64
  X_val.npy  → shape (150, 128, 128)  dtype float64
  X_test.npy  → shape (150, 128, 128)  dtype float64
  y_train.npy  → shape (699,)  dtype <U9
  y_val.npy  → shape (150,)  dtype <U9
  y_test.npy  → shape (150,)  dtype <U9

 Espectrogramas prontos.


## 5. Verificação final

In [9]:
import os
import numpy as np

print('=== VERIFICAÇÃO FINAL ===')

TAB = f'{PROJECT}/data/processed/tabular'
IMG = f'{PROJECT}/data/processed/images'

print('\nDados tabulares')
for fname in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test']:
    path = f'{TAB}/{fname}.npy'
    exists = os.path.exists(path)
    if exists:
        arr = np.load(path, allow_pickle=True)
        print(f'  ✓ {fname}.npy  shape={arr.shape}')
    else:
        print(f'  X {fname}.npy  FALTANDO')

print('\nEspectrogramas')
for fname in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test']:
    path = f'{IMG}/{fname}.npy'
    exists = os.path.exists(path)
    if exists:
        arr = np.load(path, allow_pickle=True)
        print(f'  ✓ {fname}.npy  shape={arr.shape}')
    else:
        print(f'  X {fname}.npy  FALTANDO')

=== VERIFICAÇÃO FINAL ===

Dados tabulares
  ✓ X_train.npy  shape=(700, 57)
  ✓ X_val.npy  shape=(150, 57)
  ✓ X_test.npy  shape=(150, 57)
  ✓ y_train.npy  shape=(700,)
  ✓ y_val.npy  shape=(150,)
  ✓ y_test.npy  shape=(150,)

Espectrogramas
  ✓ X_train.npy  shape=(699, 128, 128)
  ✓ X_val.npy  shape=(150, 128, 128)
  ✓ X_test.npy  shape=(150, 128, 128)
  ✓ y_train.npy  shape=(699,)
  ✓ y_val.npy  shape=(150,)
  ✓ y_test.npy  shape=(150,)
